In [1]:
import pandas as pd
import numpy as np
import os
from glob import glob

parquet_files = glob('./kaggle/input/jane-street-real-time-market-data-forecasting/train.parquet/**/*.parquet', recursive=True)

# funtion to reduce memory usage
def reduce_mem_usage(df):
    """ iterate through all the columns of a dataframe and modify the data type
        to reduce memory usage.        
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

def drop_cols(df):
    """Clean the DataFrame by dropping specified columns and handling missing values."""
    # List of columns to drop
    columns_to_drop = ['feature_27', 'feature_00', 'feature_01', 'feature_02', 
                      'feature_03', 'feature_04', 'feature_21', 'feature_26', 
                      'feature_31', 'responder_0', 'responder_1', 'responder_2', 
                      'responder_3', 'responder_4', 'responder_5', 'responder_7', 
                      'responder_8']
    
    # Drop only existing columns
    existing_columns = [col for col in columns_to_drop if col in df.columns]
    if existing_columns:
        df = df.drop(columns=existing_columns)
    
    return df

def fill_nans(df):
    # First, sort by date_id and time_id
    df = df.sort_values(['date_id', 'time_id'])
    
    # Fill NaNs within each symbol_id group
    def fill_group(group):
        for column in group.columns:
            if group[column].dtype.kind in 'iuf':  # integer, unsigned int, or float
                # Interpolate within each group
                group[column] = group[column].interpolate(method='linear', limit_direction='both')
            else:
                # Fill with mode for categorical/other types
                group[column] = group[column].fillna(group[column].mode()[0] if not group[column].mode().empty else None)
        return group

    # Apply the fill within each group of symbol_id
    df = df.groupby('symbol_id', group_keys=False).apply(fill_group)
    return df

for i, file in enumerate(sorted(parquet_files)):
    df = pd.read_parquet(file)
    df = reduce_mem_usage(df)
    df = drop_cols(df)
    df.to_parquet(f'./kaggle/working/part{i}.parquet', index=False)
    del df

files = sorted(glob('./kaggle/working/*.parquet'))
df = pd.concat([pd.read_parquet(file) for file in files])
df = fill_nans(df)
df.to_parquet('./kaggle/working/train.parquet', index=False)

Memory usage of dataframe is 654.51 MB
Memory usage after optimization is: 435.72 MB
Decreased by 33.4%
Memory usage of dataframe is 944.04 MB
Memory usage after optimization is: 548.24 MB
Decreased by 41.9%
Memory usage of dataframe is 1022.35 MB
Memory usage after optimization is: 593.72 MB
Decreased by 41.9%
Memory usage of dataframe is 1352.24 MB
Memory usage after optimization is: 693.36 MB
Decreased by 48.7%
Memory usage of dataframe is 1690.96 MB
Memory usage after optimization is: 867.04 MB
Decreased by 48.7%
Memory usage of dataframe is 1800.46 MB
Memory usage after optimization is: 923.18 MB
Decreased by 48.7%
Memory usage of dataframe is 2088.53 MB
Memory usage after optimization is: 1070.89 MB
Decreased by 48.7%
Memory usage of dataframe is 2132.85 MB
Memory usage after optimization is: 1093.61 MB
Decreased by 48.7%
Memory usage of dataframe is 2067.02 MB
Memory usage after optimization is: 1059.86 MB
Decreased by 48.7%
Memory usage of dataframe is 2112.32 MB
Memory usage a

/var/folders/wh/srwjqw_j5gsbl1y7xdb9jc900000gn/T/ipykernel_68225/3010873802.py:79: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('symbol_id', group_keys=False).apply(fill_group)
